# Knock Decision

Knocking locks in a score outcome, so there is no `position_cost` to minimise — we compare the value of cashing out now against a flat opportunity cost of playing on.

The policy returns an EV (so the eval harness can threshold it) and approximates the future with a single constant $\kappa$.

The opponent's deadwood $O$ is the missing quantity.
We model it as a distribution, sampling opponent hands consistent with the belief state. 

In [9]:
import sys, os, random
import numpy as np
sys.path.insert(0, os.path.abspath('..'))

from agent.cards import (Card, SUITS, RANKS, RANK_INDEX, make_deck,
                         find_best_melds, is_set, is_run)
from agent.inference import BeliefState

DECK = make_deck()
def deadwood(cards):      return sum(c.value for c in cards)
def best_deadwood(hand):  _, dw = find_best_melds(hand); return deadwood(dw)

## 1. When we may knock, and what we win

We decide at the moment we hold 11 cards (just drew, about to discard). Knocking is legal once the best post-discard deadwood $k \le 10$; from the 11-card hand we pick the discard that minimises $k$ and remember the melds we keep. Given the opponent's post-layoff deadwood $o^*$, the engine resolves three branches (signs from our perspective):

$$\text{gain} = \begin{cases}
O + 25 & k = 0 \quad(\textbf{gin; opponent cannot lay off})\\
o^* - k & o^* > k \quad(\textbf{knock, we win})\\
-\big((k - o^*) + 25\big) & o^* \le k \quad(\textbf{undercut, we lose})
\end{cases}$$

On a non-gin knock the opponent first lays deadwood off onto our melds, so $o^*$ is their raw deadwood *minus* whatever fits our melds. We replicate the engine's layoff loop exactly.

In [10]:
def knock_discard(hand):
    # hand = 11 cards. Choose the discard minimising kept deadwood.
    # Returns (k, discard, our_melds) for the best 10-card keep.
    best = (float('inf'), None, None)
    for d in hand:
        kept = [c for c in hand if c is not d]
        melds, dw = find_best_melds(kept)
        k = deadwood(dw)
        if k < best[0]:
            best = (k, d, melds)
    return best

def can_knock(hand):
    return knock_discard(hand)[0] <= 10

def layoff_reduce(opp_deadwood, our_melds):
    # Opponent sheds deadwood onto our melds (same fixpoint loop as game.py).
    remaining = list(opp_deadwood)
    melds = [list(m) for m in our_melds]
    changed = True
    while changed:
        changed = False
        for c in remaining[:]:
            for m in melds:
                if is_set(m + [c]) or is_run(m + [c]):
                    m.append(c); remaining.remove(c); changed = True; break
            if changed:
                break
    return deadwood(remaining)

## 2. Why a point estimate isn't enough

The obvious move is $\hat o = \mathbb{E}[O] = \sum_c p(c)\,\text{value}(c)$ and compare to $k$. Two things break it:

1. $\mathbb{E}[O]$ is the expected *pip total* of a 10-card hand (~50–65). It dwarfs any $k\le 10$, so the knock-win branch always fires and the rule says knock, always.
2. The belief is a per-card marginal. It cannot represent "this hand is three melds" — the structure that gives a near-gin opponent low deadwood — so the undercut case never appears.

We need the distribution of $O$, and the cleanest way to get it from marginals is to sample whole hands.

In [11]:
def expected_opp_deadwood_pointestimate(bs):
    return sum(bs.prob(c) * c.value for c in DECK if bs.prob(c) > 0)

# blind belief: E[O] is enormous next to any legal k -> "always knock"
_own = [Card('7','H'),Card('8','H'),Card('9','H'),Card('5','C'),Card('5','D'),
        Card('5','S'),Card('A','C'),Card('2','D'),Card('3','S'),Card('4','H')]
_bs = BeliefState(_own, Card('K','S'))
print(f"point-estimate E[O] (blind) = {expected_opp_deadwood_pointestimate(_bs):.1f}  vs  any k <= 10")

point-estimate E[O] (blind) = 68.5  vs  any k <= 10


## 3. Sampling opponent hands from the belief

We draw hands consistent with the marginals using Madow systematic sampling: it returns exactly the right number of cards every draw and reproduces each card's inclusion probability $p(c)$.

Certain cards ($p=1$) are always included; impossible cards ($p=0$) never are. Among the uncertain cards the slot count is $n = 10 - (\#\text{certain})$, which equals $\sum p_{\text{unc}}$ exactly because $\sum_c p(c)=10$. We lay the uncertain probabilities end to end (cumulative sums $C_i$), pick a random start $r\in[0,1)$, and select the cards hit by $r, r{+}1, \dots, r{+}n{-}1$. Each unit's interval has length $p(c)\le 1 <$ the spacing, so we always land on $n$ distinct cards.

Independent marginals under-represent melds (they don't enforce co-occurrence), but they do put elevated mass on the right cards, so sampled hands form melds whenever the belief concentrates on partners — which is exactly when a real opponent is near gin.

In [12]:
def sample_opp_hands(bs, n_samples, rng):
    p = np.array([bs.prob(c) for c in DECK])
    certain = p >= 1 - 1e-9
    uncertain = (p > 1e-12) & (~certain)
    ui = np.where(uncertain)[0]
    pi = p[ui]
    n_slots = int(round(pi.sum()))          # == 10 - #certain
    C = np.cumsum(pi)
    cert_cards = [DECK[i] for i in np.where(certain)[0]]
    base = np.arange(n_slots)
    starts = rng.random(n_samples)
    hands = []
    for s in range(n_samples):
        sel = np.searchsorted(C, starts[s] + base, side='left')   # interval hit by each point
        hands.append(cert_cards + [DECK[ui[j]] for j in sel])
    return hands

# sanity: every hand has 10 cards, and sampled frequencies match the belief marginals
_rng = np.random.default_rng(0)
_h = sample_opp_hands(_bs, 5000, _rng)
from collections import Counter
_c = Counter(c for hand in _h for c in hand)
print("hand sizes:", set(len(h) for h in _h))
for card in [Card('6','H'), Card('Q','D'), Card('2','C')]:
    print(f"  {card}:  belief={_bs.prob(card):.3f}  sampled={_c[card]/5000:.3f}")

hand sizes: {10}
  6H:  belief=0.244  sampled=0.248
  QD:  belief=0.244  sampled=0.241
  2C:  belief=0.244  sampled=0.244


## 4. Knock EV over the distribution

For each sampled opponent hand we compute its real deadwood (and lay it off onto our melds), evaluate the branch table, and average. We also report $P(o^*\le k)$ — the undercut probability — because the eval harness will want to watch it directly.

In [13]:
def knock_distribution(hand, bs, n_samples=4000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    k, discard, our_melds = knock_discard(hand)
    if k > 10:
        return None                      # not a legal knock
    gains, ostars = [], []
    undercut = 0
    for opp in sample_opp_hands(bs, n_samples, rng):
        _, o_dw = find_best_melds(opp)
        o_raw = deadwood(o_dw)
        if k == 0:
            gain, ostar = o_raw + 25, o_raw          # gin: no layoff
        else:
            ostar = layoff_reduce(o_dw, our_melds)
            if ostar > k:
                gain = ostar - k
            else:
                gain = -((k - ostar) + 25); undercut += 1
        gains.append(gain); ostars.append(ostar)
    gains = np.array(gains)
    return dict(k=k, discard=discard, ev=gains.mean(),
                undercut=undercut / n_samples, ostars=np.array(ostars))

def should_knock(hand, bs, kappa=0.0, n_samples=4000, rng=None):
    r = knock_distribution(hand, bs, n_samples, rng)
    return r is not None and r['ev'] > kappa

## 5. Worked position — the decision flips on belief

The same 11-card hand, knockable at $k=10$, against three readings of the opponent:

- **far** — blind belief: opponent looks like a random hand, deadwood high, undercut almost impossible → cash out;
- **medium** — they took one low run off the discard: a bit of undercut risk creeps in;
- **near** — they took two low runs: probably sitting on a tiny deadwood → knocking walks straight into an undercut.

$k$ never changes. Only the inferred opponent deadwood does, and that alone flips knock → wait.

In [18]:
hand = [Card('7','H'),Card('8','H'),Card('9','H'),     # run
        Card('5','C'),Card('5','D'),Card('5','S'),     # set
        Card('A','C'),Card('2','D'),Card('3','S'),Card('4','H'),
        Card('K','S')]                                 # 11 cards; KS is the discard
own = [c for c in hand if c is not hand[-1]]

def near_belief(taken, nu=4.0):
    b = BeliefState(own, Card('K','S'))
    for c in taken:                                    # opponent grabs low connectors
        b.observe_opponent_draw_discard_bayesian(c, nu=nu)
        b.observe_opponent_discard(Card('Q','H'))      # ...and throws a high card
    return b

beliefs = {
    'far    (blind)':        BeliefState(own, Card('K','S')),
    'medium (one low run)':  near_belief([Card('2','D'),Card('3','D'),Card('4','D')]),
    'near   (two low runs)': near_belief([Card('2','D'),Card('3','D'),Card('4','D'),
                                          Card('2','C'),Card('3','C'),Card('4','C')]),
}
res = {}
for name, b in beliefs.items():
    r = knock_distribution(hand, b, 5000, np.random.default_rng(11))
    res[name] = r
    print(f"{name:22s} belief_sum={b.hand_size_belief:4.1f}  k={r['k']}  "
          f"EV={r['ev']:6.2f}  P(undercut)={r['undercut']:5.1%}  median o*={np.median(r['ostars']):.0f}")

far    (blind)         belief_sum=10.0  k=10  EV= 53.35  P(undercut)= 0.0%  median o*=63
medium (one low run)   belief_sum=10.0  k=10  EV= 15.65  P(undercut)= 9.6%  median o*=29
near   (two low runs)  belief_sum=10.0  k=10  EV=-18.49  P(undercut)=67.0%  median o*=7


In [15]:
print("kappa sweep — same hand, decision per belief:\n")
print(f"  {'kappa':>5} | " + " | ".join(f"{n.split()[0]:>6}" for n in beliefs))
for kappa in (0, 3, 6, 10, 15):
    row = " | ".join(f"{'KNOCK' if res[n]['ev'] > kappa else 'wait':>6}" for n in beliefs)
    print(f"  {kappa:>5} | {row}")

kappa sweep — same hand, decision per belief:

  kappa |    far | medium |   near
      0 |  KNOCK |  KNOCK |   wait
      3 |  KNOCK |  KNOCK |   wait
      6 |  KNOCK |  KNOCK |   wait
     10 |  KNOCK |  KNOCK |   wait
     15 |  KNOCK |  KNOCK |   wait


## 6. The undercut tail, made visible

The point estimate saw one number; the distribution shows where the mass sits. Below, the opponent-deadwood histograms for *far* vs *near*. Everything at or below $k=10$ is an undercut — a thin sliver for *far*, the bulk of the mass for *near*.

In [16]:
def hist(ostars, k, width=40, bins=(0,5,10,15,20,25,30,40,60)):
    n = len(ostars)
    for lo, hi in zip(bins, bins[1:]):
        frac = np.mean((ostars >= lo) & (ostars < hi))
        bar = '#' * round(frac * width)
        flag = '  <- undercut' if hi <= k + 1 else ''
        print(f"  o* [{lo:2d},{hi:2d}) {frac:5.1%} {bar}{flag}")

for name in ('far    (blind)', 'near   (two low runs)'):
    print(f"{name}   (k={res[name]['k']}, P(undercut)={res[name]['undercut']:.1%}):")
    hist(res[name]['ostars'], res[name]['k'])
    print()

far    (blind)   (k=10, P(undercut)=0.0%):
  o* [ 0, 5)  0.0%   <- undercut
  o* [ 5,10)  0.0%   <- undercut
  o* [10,15)  0.0% 
  o* [15,20)  0.0% 
  o* [20,25)  0.0% 
  o* [25,30)  0.0% 
  o* [30,40)  0.0% 
  o* [40,60) 31.9% #############

near   (two low runs)   (k=10, P(undercut)=67.0%):
  o* [ 0, 5) 28.6% ###########  <- undercut
  o* [ 5,10) 25.9% ##########  <- undercut
  o* [10,15) 45.5% ##################
  o* [15,20)  0.0% 
  o* [20,25)  0.0% 
  o* [25,30)  0.0% 
  o* [30,40)  0.0% 
  o* [40,60)  0.0% 



## 7. Reduction to the greedy baseline

`sim.py` knocks the moment deadwood $\le 10$. With a blind belief the sampled $O$ sits far above any $k\le 10$, undercut is negligible, so $\text{EV}>0$ on every legal hand and `should_knock` at $\kappa=0$ fires exactly when the hand is legal — matching `sim.py`.

In [17]:
random.seed(7)
N = 150
legal = agree = 0
for _ in range(N):
    deck = make_deck(); random.shuffle(deck)
    h = deck[:11]                                  # 11-card decision hand
    bs = BeliefState(deck[:10], deck[11])          # blind belief
    baseline = can_knock(h)                        # sim.py: knock iff legal
    ours = should_knock(h, bs, kappa=0.0, n_samples=120, rng=np.random.default_rng(_))
    legal += baseline
    agree += (ours == baseline)
print(f"legal-knock hands: {legal}/{N}")
print(f"should_knock(blind, kappa=0) agrees with knock-at-first-legal: {agree}/{N}")

legal-knock hands: 0/150
should_knock(blind, kappa=0) agrees with knock-at-first-legal: 150/150


## Summary

Modelling $O$ as a distribution instead of a point estimate is what makes the knock decision real. The undercut branch now has mass, $P(o^*\le k)$ falls straight out of the sample, and the same hand at the same $k$ knocks or waits depending only on what we've inferred about the opponent. Layoff is handled exactly per sample, so we're not over-crediting the win margin either.

What's still approximate, and which way it leans:

- **Independent marginals under-count melds.** Sampling from per-card probabilities misses co-occurrence, so sampled hands have slightly more deadwood than a real near-gin opponent — we under-estimate undercut a little (mildly optimistic about knocking). A joint/meld-aware belief would tighten the tail.
- **$\kappa$ is still a flat stand-in for the future.** It should rise as the opponent nears gin and fall when we can't improve.
- **Sampling cost.** A few thousand `find_best_melds` calls per decision is fine for eval at the current scale, but if the policy is in an inner MCTS loop it'll want caching or a cheaper deadwood estimate.